# Animation inverse — rotation de la plateforme (triangle de 0,1 m)

Ce notebook indépendant anime le robot parallèle pendant que son effecteur triangulaire tourne d'un tour complet **sur lui-même**, sans déplacer son centre. Le côté du triangle vaut **0,1 m**.

Pour que la rotation complète reste réalisable avec un centre fixe, les trois biellettes entre les 5-barres et le triangle sont allongées à **0,23 m** et peuvent pivoter; leur orientation est recalculée à chaque image. Le solveur choisit une configuration proche de l'image précédente pour éviter les sauts d'angles. Les autres longueurs et les positions des moteurs fixes reprennent les conventions des notebooks précédents.

## 1 — Paramètres et points fixes

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Longueurs des barres (m)
AB = BC = CD = DE = EA = 0.2
l_triangle = 0.23  # allongement nécessaire pour un tour complet à centre fixe
d_triangle = 0.1

# Centre fixe de la plateforme et paramètres de l'animation
platform_center = np.array([0.134, 0.547])
n_frames = 72
n_connector_directions = 360#TODO
geometry_tolerance = 1e-10

# Points fixes des moteurs, avec les formules de compute_fixed_points()
A1 = np.array([0.0, 0.0])
A2 = np.array([
    (0.5 + np.cos(np.pi / 3) + 2 * np.cos(np.pi / 6)
     + np.cos(5 * np.pi / 6 - 2 * np.pi / 5)) * 0.2,
    (np.sin(2 * np.pi / 5) + np.sin(4 * np.pi / 5) + 1
     + np.sin(np.pi / 3) + np.sin(2 * np.pi / 3)
     + np.sin(2 * np.pi / 3 + 2 * np.pi / 5)) * 0.2,
])
A3 = np.array([
    -(0.5 + np.cos(np.pi / 3) + np.cos(np.pi / 6)
      + np.cos(4 * np.pi / 5 - np.pi / 3
               + np.cos(2 * np.pi / 5 - np.pi / 3))) * 0.2,
    A2[1] + np.sin(2 * np.pi / 3) * 0.2,
])

base_angles = np.array([0.0, 2 * np.pi / 3, -2 * np.pi / 3])
left_anchors = np.array([A1, A2, A3])
right_anchors = left_anchors + EA * np.column_stack((
    np.cos(base_angles), np.sin(base_angles)
))
print("Points fixes des trois 5-barres calculés.")

Points fixes des trois 5-barres calculés.


## 2 — Solveur géométrique et suivi des branches

À chaque image, chaque biellette de connexion est essayée dans 360 directions. Pour chaque position candidate de l'effecteur, le solveur trouve les intersections de cercles des deux 5-barres. Il retient la solution qui change le moins les angles moteurs et la direction de biellette par rapport à l'image précédente.

In [2]:
def rotate(vector, angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([c * vector[0] - s * vector[1],
                     s * vector[0] + c * vector[1]])

def cross_2d(vector_0, vector_1):
    return vector_0[0] * vector_1[1] - vector_0[1] * vector_1[0]

def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def global_to_local(point, left_anchor, base_angle):
    return rotate(np.asarray(point) - left_anchor, -base_angle)

def local_to_global(point, left_anchor, base_angle):
    return left_anchor + rotate(point, base_angle)

def circle_intersections(center_0, radius_0, center_1, radius_1):
    delta = center_1 - center_0
    distance = np.linalg.norm(delta)
    tol = geometry_tolerance
    if distance <= tol or distance > radius_0 + radius_1 + tol:
        return []
    if distance < abs(radius_0 - radius_1) - tol:
        return []

    along = (radius_0**2 - radius_1**2 + distance**2) / (2 * distance)
    height_squared = radius_0**2 - along**2
    if height_squared <= tol:
        return []

    unit = delta / distance
    perpendicular = np.array([-unit[1], unit[0]])
    foot = center_0 + along * unit
    height = np.sqrt(height_squared)
    return [foot + height * perpendicular, foot - height * perpendicular]

def leg_candidates(endpoint_global, leg_index):
    endpoint_local = global_to_local(
        endpoint_global, left_anchors[leg_index], base_angles[leg_index]
    )
    left_elbows = circle_intersections(
        np.array([0.0, 0.0]), AB, endpoint_local, BC
    )
    right_elbows = circle_intersections(
        np.array([EA, 0.0]), DE, endpoint_local, CD
    )
    candidates = []

    for elbow_left in left_elbows:
        for elbow_right in right_elbows:
            assembly_sign = cross_2d(
                elbow_right - elbow_left, endpoint_local - elbow_left
            )
            if abs(assembly_sign) <= geometry_tolerance:
                continue
            motor_angles = np.array([
                np.arctan2(elbow_left[1], elbow_left[0]),
                np.arctan2(elbow_right[1], elbow_right[0] - EA),
            ])
            candidates.append({
                "endpoint_local": endpoint_local,
                "elbow_left_local": elbow_left,
                "elbow_right_local": elbow_right,
                "motor_angles": motor_angles,
                "assembly_sign": np.sign(assembly_sign),
            })
    return candidates

def choose_leg_state(vertex, leg_index, previous_state, first_frame):
    best_state = None
    best_score = np.inf
    connector_angles = np.linspace(
        -np.pi, np.pi, n_connector_directions, endpoint=False
    )

    for connector_angle in connector_angles:
        endpoint = vertex + l_triangle * np.array([
            np.cos(connector_angle), np.sin(connector_angle)
        ])
        for candidate in leg_candidates(endpoint, leg_index):
            if first_frame:
                motor_delta = wrap_angle(
                    candidate["motor_angles"] - np.deg2rad([90.0, 90.0])
                )
                score = np.sum(motor_delta**2)
            else:
                motor_delta = wrap_angle(
                    candidate["motor_angles"] - previous_state["motor_angles"]
                )
                connector_delta = wrap_angle(
                    connector_angle - previous_state["connector_angle"]
                )
                assembly_penalty = (
                    0.0 if candidate["assembly_sign"] == previous_state["assembly_sign"]
                    else 10.0
                )
                score = (
                    np.sum(motor_delta**2)
                    + 0.2 * connector_delta**2
                    + assembly_penalty
                )

            if score < best_score:
                best_score = score
                best_state = {
                    **candidate,
                    "vertex": vertex.copy(),
                    "endpoint_global": endpoint,
                    "connector_angle": connector_angle,
                }

    if best_state is None:
        raise ValueError(
            f"Aucune configuration atteignable pour la jambe {leg_index + 1}. "
            "Essayez un autre centre de rotation ou davantage de directions candidates."
        )
    return best_state

## 3 — Calcul des images et vérification des longueurs

In [3]:
# Par rapport au centroïde, les sommets A, B, C commencent à -150, -30 et 90 degrés.
vertex_phases = np.deg2rad([-150.0, -30.0, 90.0])
vertex_radius = d_triangle / np.sqrt(3)
rotation_angles = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False)

frames = []
previous_states = [None, None, None]
for frame_index, alpha in enumerate(rotation_angles):
    vertices = np.array([
        platform_center + rotate(
            vertex_radius * np.array([np.cos(phase), np.sin(phase)]), alpha
        )
        for phase in vertex_phases
    ])
    current_states = [
        choose_leg_state(
            vertices[i], i, previous_states[i], first_frame=(frame_index == 0)
        )
        for i in range(3)
    ]

    for i, state in enumerate(current_states):
        state["elbow_left_global"] = local_to_global(
            state["elbow_left_local"], left_anchors[i], base_angles[i]
        )
        state["elbow_right_global"] = local_to_global(
            state["elbow_right_local"], left_anchors[i], base_angles[i]
        )
    frames.append({"alpha": alpha, "vertices": vertices, "legs": current_states})
    previous_states = current_states

# Check platform dimensions, connector lengths, and all four 5-bar links.
for frame_index, frame in enumerate(frames):
    vertices = frame["vertices"]
    side_lengths = [
        np.linalg.norm(vertices[i] - vertices[(i + 1) % 3])
        for i in range(3)
    ]
    if not np.allclose(side_lengths, d_triangle, atol=1e-8, rtol=0):
        raise AssertionError(f"Triangle incorrect à l'image {frame_index}.")

    for leg_index, state in enumerate(frame["legs"]):
        c = state["elbow_left_local"]
        d = state["elbow_right_local"]
        e = state["endpoint_local"]
        lengths = [
            np.linalg.norm(c),
            np.linalg.norm(e - c),
            np.linalg.norm(e - d),
            np.linalg.norm(d - np.array([EA, 0.0])),
            np.linalg.norm(state["endpoint_global"] - state["vertex"]),
        ]
        expected = [AB, BC, CD, DE, l_triangle]
        if not np.allclose(lengths, expected, atol=1e-8, rtol=0):
            raise AssertionError(
                f"Longueurs incorrectes à l'image {frame_index}, jambe {leg_index + 1}."
            )

all_motor_angles = np.array([
    [state["motor_angles"] for state in frame["legs"]]
    for frame in frames
])
print(f"{len(frames)} images vérifiées; rotation = 360°; côté du triangle = {d_triangle:.2f} m.")
print("Plages d'angles moteurs (degrés), jambes x gauche/droite :")
print(np.round(np.degrees(all_motor_angles).min(axis=0), 1))
print(np.round(np.degrees(all_motor_angles).max(axis=0), 1))

72 images vérifiées; rotation = 360°; côté du triangle = 0.10 m.
Plages d'angles moteurs (degrés), jambes x gauche/droite :
[[79.6 61.7]
 [72.1 53.3]
 [81.7 62.1]]
[[113.9  84.1]
 [ 91.9 100.7]
 [119.1  80.1]]


## 4 — Animation

Exécuter la cellule pour afficher le tour complet de la plateforme. Le centroïde reste fixe; les trois biellettes de connexion s'orientent librement afin que chaque image reste compatible avec la géométrie des 5-barres.

In [4]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = ["tab:blue", "tab:orange", "tab:green"]
leg_lines = [
    ax.plot([], [], "o-", color=colors[i], linewidth=2,
            label=f"5-barre {i + 1}")[0]
    for i in range(3)
]
connector_lines = [
    ax.plot([], [], "--", color=colors[i], linewidth=1.5,
            label="Biellette de connexion" if i == 0 else None)[0]
    for i in range(3)
]
triangle_line, = ax.plot([], [], "k-o", linewidth=2, label="Triangle 0,1 m")
ax.scatter(left_anchors[:, 0], left_anchors[:, 1], marker="s", color="black", zorder=4)
ax.scatter(right_anchors[:, 0], right_anchors[:, 1], marker="s", color="black", zorder=4)
ax.set_title("Rotation sur place — effecteur triangulaire")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.45, 0.75)
ax.set_ylim(-0.15, 0.9)
ax.grid(True)
ax.legend(loc="upper right")
angle_label = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top")

def update(frame_index):
    frame = frames[frame_index]
    for i, state in enumerate(frame["legs"]):
        points = np.array([
            left_anchors[i],
            state["elbow_left_global"],
            state["endpoint_global"],
            state["elbow_right_global"],
            right_anchors[i],
        ])
        leg_lines[i].set_data(points[:, 0], points[:, 1])
        connector_points = np.array([state["endpoint_global"], state["vertex"]])
        connector_lines[i].set_data(connector_points[:, 0], connector_points[:, 1])

    closed_triangle = np.vstack([frame["vertices"], frame["vertices"][0]])
    triangle_line.set_data(closed_triangle[:, 0], closed_triangle[:, 1])
    angle_label.set_text(f"Rotation : {np.degrees(frame['alpha']):.0f}°")
    return [*leg_lines, *connector_lines, triangle_line, angle_label]

animation = FuncAnimation(
    fig, update, frames=len(frames), interval=80, blit=True, repeat=True
)
plt.close(fig)
display(HTML(animation.to_jshtml()))